In [1]:
from scipy.stats import norm
import numpy as np
import time
import import_ipynb
#from Geometric_Brownian_Motion import simulate_GBM, simulate_GBM_antithetic
#from Black_Scholes_Closed_Form import european_mc, european_mc_antithetic
import matplotlib.pyplot as plt
from scipy.stats import qmc

In [6]:
S0, K, r, sigma, T = 100, 100, 0.05, 0.2, 1.0

In [8]:
def pathwise_delta(S0, K, r, sigma, T, n_paths, type="call"):
    Z = np.random.normal(0, 1, n_paths)
    S_T = S0 * np.exp((r - 0.5*sigma**2)*T + sigma*np.sqrt(T)*Z)
    
    if type == "call":
        indicator = (S_T > K).astype(float)
    else:
        indicator = (S_T < K).astype(float)
        # for a put, d(payoff)/dS_T = -1{S_T < K}, so overall sign flips
    
    sign = 1 if type == "call" else -1
    delta_paths = np.exp(-r*T) * sign * indicator * (S_T / S0)
    
    delta = delta_paths.mean()
    se = delta_paths.std(ddof=1) / np.sqrt(n_paths)
    return delta, se

In [9]:
def pathwise_vega(S0, K, r, sigma, T, n_paths, type="call"):
    Z = np.random.normal(0, 1, n_paths)
    S_T = S0 * np.exp((r - 0.5*sigma**2)*T + sigma*np.sqrt(T)*Z)
    
    if type == "call":
        indicator = (S_T > K).astype(float)
    else:
        indicator = (S_T < K).astype(float)
    
    dS_T_dsigma = S_T * (np.sqrt(T)*Z - sigma*T)
    vega_paths = np.exp(-r*T) * indicator * dS_T_dsigma
    
    vega = vega_paths.mean()
    se = vega_paths.std(ddof=1) / np.sqrt(n_paths)
    return vega, se

In [10]:
def bs_delta(S0, K, r, sigma, T, type="call"):
    d1 = (np.log(S0/K) + (r + 0.5*sigma**2)*T) / (sigma*np.sqrt(T))
    if type == "call":
        return norm.cdf(d1)
    else:
        return norm.cdf(d1) - 1

def bs_vega(S0, K, r, sigma, T):
    d1 = (np.log(S0/K) + (r + 0.5*sigma**2)*T) / (sigma*np.sqrt(T))
    return S0 * norm.pdf(d1) * np.sqrt(T)

In [11]:
delta_pw, se_delta = pathwise_delta(S0, K, r, sigma, T, n_paths=100000, type="call")
vega_pw, se_vega = pathwise_vega(S0, K, r, sigma, T, n_paths=100000, type="call")

print(f"Pathwise Delta: {delta_pw:.4f} ± {1.96*se_delta:.4f}")
print(f"BS Delta:       {bs_delta(S0, K, r, sigma, T, 'call'):.4f}")
print(f"Pathwise Vega:  {vega_pw:.4f} ± {1.96*se_vega:.4f}")
print(f"BS Vega:        {bs_vega(S0, K, r, sigma, T):.4f}")

Pathwise Delta: 0.6359 ± 0.0036
BS Delta:       0.6368
Pathwise Vega:  37.7739 ± 0.4732
BS Vega:        37.5240
